**Model:** Stable Diffusion v1.5 + Dual ControlNet (Canny + Segmentation) + LoRA fine-tune  
**Task:** Transfer time-of-day and weather conditions to an input image guided by a text prompt.

---

1. **Run Setup** (Step 1) — installs packages and checks  GPU  
2. **Upload images and enter a prompt** (Section 2)  
3. **Run all remaining cells top-to-bottom**
4. **Download the zip** (Section 8) — contains the generated image and preprocessed target for evaluation

**Tested on  runtime:** GPU (L4 or better).

---

## Section 1 — Setup and Installation

In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.44.0 peft==0.12.0 accelerate==0.34.0
!pip install -q pillow opencv-python-headless matplotlib gdown

import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


---
## Section 2 — Inference Parameters
Pre-set to match the training evaluation. Adjust only if experimenting.

In [ ]:
import os

IMG_SIZE       = (512, 512)
STRENGTH       = 0.80
NUM_STEPS      = 30
GUIDANCE_SCALE = 9.0
CANNY_SCALE    = 0.65
SEG_SCALE      = 0.70

OUTPUT_DIR = "/content/inference_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16

print("Parameters configured")
print(f"  Device: {DEVICE}  |  Image size: {IMG_SIZE}")
print(f"  Strength: {STRENGTH}  |  Steps: {NUM_STEPS}  |  Guidance: {GUIDANCE_SCALE}")
print(f"  Canny scale: {CANNY_SCALE}  |  Seg scale: {SEG_SCALE}")

---
## Section 3 — Download LoRA Weights

In [ ]:
import gdown

LORA_DIR         = "/content/lora_weights"
GDRIVE_FOLDER_ID = "1i3fQelMbLIr7q6HDM0JJUKU-Rkglp3KY"

os.makedirs(LORA_DIR, exist_ok=True)

print("Downloading LoRA weights from Google Drive...")
gdown.download_folder(
    id=GDRIVE_FOLDER_ID,
    output=LORA_DIR,
    quiet=False,
    use_cookies=False,
)

# Locate adapter_config.json in case weights landed in a subfolder
LORA_PATH = LORA_DIR
for root, dirs, fnames in os.walk(LORA_DIR):
    if "adapter_config.json" in fnames:
        LORA_PATH = root
        break

print(f"\nLoRA weights ready at: {LORA_PATH}")
print("Files:", os.listdir(LORA_PATH))

---
## Section 4 — Load Model Pipeline
Downloads ~6 GB of base model weights on first run (cached afterwards).

In [ ]:
from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    UNet2DConditionModel,
    UniPCMultistepScheduler,
)
from peft import PeftModel

print("Loading model pipeline (~6 GB on first run)...")
print("-" * 60)

print("[1/4] ControlNet — Canny edge...")
controlnet_canny = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=DTYPE
)

print("[2/4] ControlNet — Segmentation...")
controlnet_seg = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-seg", torch_dtype=DTYPE
)

print("[3/4] Base UNet + LoRA adapter...")
base_unet = UNet2DConditionModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", subfolder="unet", torch_dtype=DTYPE
)
unet = PeftModel.from_pretrained(base_unet, LORA_PATH)
unet.eval()

print("[4/4] Assembling Img2Img pipeline...")
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    unet=unet,
    controlnet=[controlnet_canny, controlnet_seg],
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=False)

print("-" * 60)
print("Model pipeline loaded.")

---
## Section 5 — Upload Images and Enter Prompt

**Edit the `PROMPT` variable below**, then run the cell.  
Two file pickers will appear — upload your **source image** first, then the **target (ground truth) image**.

- **Source image:** the scene you want to transform  
- **Target image:** the ground truth result (preprocessed to 512x512 and included in the download for LPIPS / condition accuracy evaluation)

### Prompt tips
Describe the target lighting and weather you want the scene transformed into.  
Example:
- `"night time,cloudy skies,heavy rain, wet reflective pavement, towne building, upenn"`

### Important Note
Please note that the model was trained on words as prompts. i.e. having words separated by commas will have better inference
Eg: sunny, clear blue skies, shadows, towne building will show much better performance than sunny day at towne

In [ ]:
from google.colab import files
from PIL import Image
import io
import matplotlib.pyplot as plt

print("Step 1 of 3: Upload your SOURCE image (the scene to transform):")
up_source = files.upload()
if not up_source:
    raise RuntimeError("No source image uploaded.")
source_filename = list(up_source.keys())[0]
INPUT_IMAGE = Image.open(io.BytesIO(up_source[source_filename])).convert("RGB")
print(f"  Source loaded: {source_filename}  ({INPUT_IMAGE.size[0]}x{INPUT_IMAGE.size[1]} px)")

print("\nStep 2 of 3: Upload your TARGET image (ground truth):")
up_target = files.upload()
if not up_target:
    raise RuntimeError("No target image uploaded.")
target_filename = list(up_target.keys())[0]
TARGET_IMAGE_RAW = Image.open(io.BytesIO(up_target[target_filename])).convert("RGB")
print(f"  Target loaded: {target_filename}  ({TARGET_IMAGE_RAW.size[0]}x{TARGET_IMAGE_RAW.size[1]} px)")

# fig, axes = plt.subplots(1, 2, figsize=(9, 4))
# axes[0].imshow(INPUT_IMAGE);      axes[0].set_title("Source Image",              fontsize=12); axes[0].axis("off")
# axes[1].imshow(TARGET_IMAGE_RAW); axes[1].set_title("Target Image (Ground Truth)", fontsize=12); axes[1].axis("off")
# plt.tight_layout()
# plt.show()

print("\nStep 3 of 3: Enter your prompt below and press Enter:")
PROMPT = input("Prompt: ").strip()
if not PROMPT:
    raise RuntimeError("No prompt entered.")
print(f"\nPrompt set: \"{PROMPT}\"")

---
## Section 6 — Preprocess Images

Both source and target are centre-cropped to 512x512 using the same pipeline as training.  
The source is also used to generate the Canny edge map and semantic segmentation map.

In [ ]:
import numpy as np
import cv2
from PIL import ImageOps
import gc

def preprocess_image(img):
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    img = ImageOps.fit(img, IMG_SIZE, method=Image.Resampling.LANCZOS)
    return img

def generate_canny(img):
    arr  = np.array(img)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    brightness = np.mean(gray)
    if brightness < 70:
        low, high = 35, 90
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    else:
        low, high = 80, 160
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    return cv2.Canny(blurred, low, high)

print("Preprocessing images...")
source_img = preprocess_image(INPUT_IMAGE)
target_img = preprocess_image(TARGET_IMAGE_RAW)

print("Generating Canny edge map...")
canny_map = generate_canny(source_img)
canny_pil = Image.fromarray(canny_map).convert("RGB")

from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation

print("Loading OneFormer segmentation model (~200 MB on first run)...")
seg_processor = OneFormerProcessor.from_pretrained("shi-labs/oneformer_ade20k_swin_tiny")
seg_model = OneFormerForUniversalSegmentation.from_pretrained(
    "shi-labs/oneformer_ade20k_swin_tiny", torch_dtype=DTYPE
).to(DEVICE)
seg_model.eval()

@torch.no_grad()
def generate_segmentation(img):
    img_r  = img.resize(IMG_SIZE, Image.Resampling.LANCZOS)
    inputs = seg_processor(images=img_r, task_inputs=["semantic"], return_tensors="pt")
    inputs = {k: v.to(DEVICE, dtype=DTYPE) if isinstance(v, torch.Tensor) else v
              for k, v in inputs.items()}
    outputs = seg_model(**inputs)
    seg_map = seg_processor.post_process_semantic_segmentation(
        outputs, target_sizes=[IMG_SIZE]
    )[0]
    seg_np = seg_map.cpu().numpy().astype(np.uint8)
    if seg_np.max() > 0:
        seg_np = (seg_np / seg_np.max() * 255).astype(np.uint8)
    return seg_np

print("Generating segmentation map...")
seg_map = generate_segmentation(source_img)
seg_pil = Image.fromarray(seg_map, mode="L").convert("RGB")

# Free seg model VRAM before loading diffusion pipeline
del seg_model, seg_processor
torch.cuda.empty_cache()
gc.collect()
print("Segmentation model freed from VRAM")

# Save preprocessed images with original filenames preserved
src_stem = os.path.splitext(source_filename)[0]
tgt_stem = os.path.splitext(target_filename)[0]
source_img.save(os.path.join(OUTPUT_DIR, f"{src_stem}-src.png"))
target_img.save(os.path.join(OUTPUT_DIR, f"{tgt_stem}-target.png"))


# Preview
# fig, axes = plt.subplots(1, 4, figsize=(20, 5))
# axes[0].imshow(source_img);  axes[0].set_title("Source (512x512)",  fontsize=11, fontweight="bold"); axes[0].axis("off")
# axes[1].imshow(canny_map, cmap="gray"); axes[1].set_title("Canny Map", fontsize=11, fontweight="bold"); axes[1].axis("off")
# axes[2].imshow(seg_pil);     axes[2].set_title("Segmentation Map",  fontsize=11, fontweight="bold"); axes[2].axis("off")
# axes[3].imshow(target_img);  axes[3].set_title("Target (512x512)",  fontsize=11, fontweight="bold"); axes[3].axis("off")
# fig.suptitle("Preprocessed Conditioning Signals", fontsize=13, fontweight="bold")
# plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, "preprocessed_grid.png"), dpi=150, bbox_inches="tight")
# plt.show()
print("Preprocessing complete.")

---
## Section 7 — Run Inference

In [ ]:
print(f"Running inference...")
print(f"Prompt: \"{PROMPT}\"")
print("-" * 60)

with torch.no_grad():
    generated = pipe(
        prompt=PROMPT,
        image=source_img,
        control_image=[canny_pil, seg_pil],
        strength=STRENGTH,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=[CANNY_SCALE, SEG_SCALE],
    ).images[0]

generated.save(os.path.join(OUTPUT_DIR, f"{src_stem}-generated.png"))
print("Generation complete.")

# Result grid: Source | Canny | Seg | Generated | Target
fig, axes = plt.subplots(1, 5, figsize=(26, 6))

axes[0].imshow(source_img);          axes[0].set_title("Source",           fontsize=12, fontweight="bold"); axes[0].axis("off")
axes[1].imshow(canny_map, cmap="gray"); axes[1].set_title("Canny Map",     fontsize=12, fontweight="bold"); axes[1].axis("off")
axes[2].imshow(seg_pil);             axes[2].set_title("Seg Map",           fontsize=12, fontweight="bold"); axes[2].axis("off")
axes[3].imshow(generated);           axes[3].set_title("Generated",         fontsize=12, fontweight="bold"); axes[3].axis("off")
axes[4].imshow(target_img);          axes[4].set_title("Target (512x512)",  fontsize=12, fontweight="bold"); axes[4].axis("off")

max_chars = 90
display_prompt = (PROMPT[:max_chars] + "...") if len(PROMPT) > max_chars else PROMPT
fig.suptitle(f'Prompt: "{display_prompt}"', fontsize=11, y=1.02)

plt.tight_layout()
plt.show()

print("-" * 60)
print("Output files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"  {fname}  ({os.path.getsize(fpath) / 1024:.0f} KB)")
print("-" * 60)
print("Run Section 8 to download all outputs as a zip.")

---
## Section 8 — Download All Results

Downloads a zip file containing:
- `generated.png` — model output  
- `target_preprocessed.png` — ground truth at 512x512 (use this for LPIPS and condition accuracy)  
- `source_preprocessed.png` — source at 512x512  

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/inference_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

print("Downloading inference_results.zip...")
files.download(zip_path + ".zip")
print("Download started.")